# Semaine 3 — Transformation, agrégation temporelle, features engineering

**Livrable S3 :** Dataset final prêt à analyser.

Ce notebook reprend `data/processed/era5_daily.parquet` (sortie de `preprocessing.py`) et y ajoute les **features dérivées** nécessaires pour répondre à la question scientifique :

> *Comment les vagues de chaleur et le stress hydrique en Tunisie et en Méditerranée ont-ils évolué entre 1980 et 2024 ?*

## Migration R → Python — features

| Traitement R (S1) | Code R | Équivalent Python |
|---|---|---|
| Variable saisonnière | `case_when(month %in% c(6,7,8) ~ 'JJA')` | `df['month'].map({...})` |
| Décennie | `(year %/% 10) * 10` | `(df.year // 10) * 10` |
| Moyenne mobile | `zoo::rollmean(x, k=10)` | `df[v].rolling(window=10).mean()` |
| Cumul saisonnier | `group_by(year, season) %>% summarise(sum = sum(tp))` | `df.groupby(['year','season'])['tp'].sum()` |
| Anomalie | `x - mean(x[1980:2010])` | `df[v] - df.loc[df.year<=2010, v].mean()` |
| Indicateur seuil | `sum(t > 30)` | `(df.t2m > 30).groupby(...).sum()` |

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path('..').resolve()
DATA = PROJECT_ROOT / 'data' / 'processed' / 'era5_daily.parquet'

df = pd.read_parquet(DATA)
df['date'] = pd.to_datetime(df['date'])
df['decade'] = (df['year'] // 10) * 10
print(f'Lignes  : {len(df):,}')
print(f'Période : {df.date.min().date()} → {df.date.max().date()}')
print(f'Zones   : {df.zone.unique().tolist()}')
df.head()

Lignes  : 32,874
Période : 1980-01-01 → 2024-12-31
Zones   : ['mediterranee', 'tunisie']


,date,year,month,season,zone,t2m,d2m,rh,tp,decade
0,1980-01-01 12:00:00,1980,1,DJF,mediterranee,11.686298,3.688222,61.625305,0.127247,1980
1,1980-01-02 12:00:00,1980,1,DJF,mediterranee,11.226440,3.159787,61.316261,0.097736,1980
2,1980-01-03 12:00:00,1980,1,DJF,mediterranee,10.444486,2.543439,61.509197,0.133025,1980
3,1980-01-04 12:00:00,1980,1,DJF,mediterranee,9.700102,1.738042,60.578163,0.092826,1980
4,1980-01-05 12:00:00,1980,1,DJF,mediterranee,9.289001,1.362613,60.960831,0.099035,1980


## 1. Indicateur — Jours de canicule (T2m > 30 °C) par année et zone

Définition retenue : un "jour chaud" est un jour où la température à 12 UTC dépasse 30 °C. C'est un proxy simple, lisible pour la soutenance, et reproductible.

In [2]:
hot = (df.assign(is_hot=df['t2m'] > 30)
         .groupby(['year', 'zone'])['is_hot']
         .sum()
         .reset_index(name='n_jours_chauds'))
hot.head()

,year,zone,n_jours_chauds
0,1980,mediterranee,0
1,1980,tunisie,81
2,1981,mediterranee,0
3,1981,tunisie,85
4,1982,mediterranee,0


## 2. Cumul saisonnier des précipitations (mm)

In [3]:
rain = (df.groupby(['year', 'season', 'zone'])['tp']
          .sum()
          .reset_index(name='tp_cumul'))
rain.head()

,year,season,zone,tp_cumul
0,1980,DJF,mediterranee,6.784879
1,1980,DJF,tunisie,3.646584
2,1980,JJA,mediterranee,3.062807
3,1980,JJA,tunisie,0.231120
4,1980,MAM,mediterranee,7.722138


## 3. Moyennes annuelles + moyenne mobile 10 ans (lissage)

In [4]:
annual = (df.groupby(['year', 'zone'])[['t2m', 'd2m', 'rh', 'tp']]
            .mean()
            .reset_index())
for v in ['t2m', 'rh', 'tp']:
    annual[f'{v}_ma10'] = (annual.groupby('zone')[v]
                                  .transform(lambda s: s.rolling(10, min_periods=3).mean()))
annual.head(15)

,year,zone,t2m,d2m,rh,tp,t2m_ma10,rh_ma10,tp_ma10
0,1980,mediterranee,19.016253,8.476663,56.769920,0.063253,NaN,NaN,NaN
1,1980,tunisie,22.611500,8.480412,45.902573,0.028486,NaN,NaN,NaN
2,1981,mediterranee,19.584183,8.681111,55.864784,0.059999,NaN,NaN,NaN
3,1981,tunisie,23.518129,8.611581,44.244667,0.017437,NaN,NaN,NaN
4,1982,mediterranee,19.328074,8.757549,56.578148,0.061785,19.309503,56.404284,0.061679
5,1982,tunisie,23.414240,9.239740,46.869717,0.034984,23.181290,45.672319,0.026969
6,1983,mediterranee,19.445965,8.611165,55.899857,0.057319,19.343618,56.278177,0.060589
7,1983,tunisie,23.432060,8.798823,45.129314,0.018863,23.243982,45.536568,0.024942
8,1984,mediterranee,19.118774,8.268264,56.162594,0.061626,19.298650,56.255061,0.060796
9,1984,tunisie,22.823063,8.382383,45.795448,0.031575,23.159798,45.588344,0.026269


## 4. Anomalies par rapport à la période de référence 1981–2010

Pratique standard OMM : la "climatologie" est calculée sur 30 ans glissants. On utilise 1981–2010 (référence WMO encore largement utilisée pour ERA5).

In [5]:
REF_PERIOD = (1981, 2010)
ref = annual[(annual.year >= REF_PERIOD[0]) & (annual.year <= REF_PERIOD[1])]
baseline = ref.groupby('zone')[['t2m', 'rh', 'tp']].mean().add_suffix('_ref')
anom = annual.merge(baseline, on='zone')
for v in ['t2m', 'rh', 'tp']:
    anom[f'{v}_anom'] = anom[v] - anom[f'{v}_ref']
anom[['year', 'zone', 't2m', 't2m_ref', 't2m_anom']].head()

,year,zone,t2m,t2m_ref,t2m_anom
0,1980,mediterranee,19.016253,19.853678,-0.837425
1,1980,tunisie,22.611500,23.840921,-1.229422
2,1981,mediterranee,19.584183,19.853678,-0.269495
3,1981,tunisie,23.518129,23.840921,-0.322792
4,1982,mediterranee,19.328074,19.853678,-0.525604


## 5. Sauvegarde du dataset enrichi (livrable S3)

In [6]:
OUT_DIR = PROJECT_ROOT / 'data' / 'processed'
anom.to_parquet(OUT_DIR / 'era5_annual_features.parquet', index=False)
hot.to_parquet(OUT_DIR / 'era5_hot_days.parquet', index=False)
rain.to_parquet(OUT_DIR / 'era5_seasonal_rain.parquet', index=False)
print('Sauvegardes :')
for p in ['era5_annual_features', 'era5_hot_days', 'era5_seasonal_rain']:
    f = OUT_DIR / f'{p}.parquet'
    print(f'  {f.name:35s} {f.stat().st_size/1e3:>6.1f} KB')

Sauvegardes :
  era5_annual_features.parquet          13.8 KB
  era5_hot_days.parquet                  2.9 KB
  era5_seasonal_rain.parquet             4.9 KB


## 6. Validation du livrable Semaine 3

| Item | Statut |
|---|---|
| Variables de saison + décennie | ✅ |
| Indicateur jours > 30 °C | ✅ `era5_hot_days.parquet` |
| Cumul saisonnier précipitations | ✅ `era5_seasonal_rain.parquet` |
| Moyennes annuelles + MA(10 ans) | ✅ `era5_annual_features.parquet` |
| Anomalies vs 1981–2010 (référence WMO) | ✅ |